In [1]:
!pip install --upgrade transformers sentencepiece sentence-transformers networkx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 78.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.3/571.3 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 646.8/646.8 kB 7.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 96.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.2.3
    Uninstalling sentence-transformers-

In [2]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

print("="*60)
print(" BƯỚC 1: TẢI MÔ HÌNH VÀO BỘ NHỚ (CHẠY 1 LẦN DUY NHẤT) ")
print("="*60)

# 1. Xác định thiết bị (Tối ưu dùng GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Đang sử dụng thiết bị: {device.type.upper()}")

# 2. Tải mô hình SBERT (Cho TextRank)
sbert_name = 'keepitreal/vietnamese-sbert'
print(f"[*] Đang tải SBERT ({sbert_name})...")
sbert_model = SentenceTransformer(sbert_name)

# 3. Tải mô hình Qwen 3.5B Instruct (Làm mượt văn bản)
qwen_name = "Qwen/Qwen2.5-3B-Instruct"
print(f"[*] Đang tải Qwen ({qwen_name})... Quá trình này sẽ mất vài phút.")
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name, trust_remote_code=True)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)

print("\n[+] TẢI MÔ HÌNH HOÀN TẤT! CÁC BIẾN TOÀN CỤC ĐÃ SẴN SÀNG.")
print("Bạn có thể chuyển sang chạy Cell 2.")

 BƯỚC 1: TẢI MÔ HÌNH VÀO BỘ NHỚ (CHẠY 1 LẦN DUY NHẤT) 
[*] Đang sử dụng thiết bị: CUDA
[*] Đang tải SBERT (keepitreal/vietnamese-sbert)...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


[+] TẢI MÔ HÌNH HOÀN TẤT! CÁC BIẾN TOÀN CỤC ĐÃ SẴN SÀNG.
Bạn có thể chuyển sang chạy Cell 2.


In [3]:
import json
import re
import torch
import networkx as nx
from sentence_transformers import util
import math
import textwrap

# ==========================================
# 1. CÁC HÀM TIỀN XỬ LÝ (NỐI TEXT & RE-CHUNK)
# ==========================================
def clean_markdown(text):
    """Xóa Markdown để chuyển về text bình thường."""
    text = re.sub(r'^#+\s+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*]+)\*', r'\1', text)
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)
    text = re.sub(r'```.*?```', '', text, flags=re.DOTALL)
    text = re.sub(r'\n{3,}', '\n\n', text).strip()
    return text

def create_uniform_chunks(text, max_tokens=150):
    """Chia lại text thành các đoạn đều nhau."""
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks, current_chunk, current_count = [], [], 0
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence: continue
        word_count = len(sentence.split())
        if current_count + word_count > max_tokens and current_count > 0:
            chunks.append({"chunk_index": len(chunks), "text": " ".join(current_chunk), "token_count": current_count})
            current_chunk = [sentence]
            current_count = word_count
        else:
            current_chunk.append(sentence)
            current_count += word_count
    if current_chunk:
        chunks.append({"chunk_index": len(chunks), "text": " ".join(current_chunk), "token_count": current_count})
    return chunks

# ==========================================
# 2. CÁC HÀM LÕI (TÍNH K, TEXTRANK & QWEN)
# ==========================================
def calculate_dynamic_k(target_words, chunk_size=150, multiplier=1.):
    """Tính số chunk K cần thiết dựa trên số từ người dùng muốn tóm tắt."""
    # Ví dụ: Muốn tóm tắt 500 từ -> Cần cấp cho model 500 * 1.5 = 750 từ đầu vào
    # Số chunk k = 750 / 150 = 5 chunks
    required_input_words = target_words * multiplier
    k = math.ceil(required_input_words / chunk_size)
    return k

def get_top_k_chunks(chunks, k):
    print(f"[-] Đang chạy TextRank để nhặt ra Top {k} chunks quan trọng nhất...")
    texts = [c['text'] for c in chunks]
    embeddings = sbert_model.encode(texts, show_progress_bar=False) # Dùng SBERT từ Cell 1
    similarity_matrix = util.cos_sim(embeddings, embeddings).numpy()
    
    nx_graph = nx.from_numpy_array(similarity_matrix)
    scores = nx.pagerank(nx_graph)
    
    for i, chunk in enumerate(chunks):
        chunk['score'] = scores[i]
        
    chunks.sort(key=lambda x: x['score'], reverse=True)
    top_k_chunks = chunks[:k]
    top_k_chunks.sort(key=lambda x: x['chunk_index']) # Giữ đúng thứ tự logic bài báo
    return top_k_chunks

def rewrite_with_qwen(top_chunks, target_words):
    combined_text = " ".join([c['text'] for c in top_chunks])
    
    # 1. Tối ưu Prompt: Ép viết chi tiết để đủ chữ
    messages = [
        {
            "role": "system", 
            "content": f"Bạn là một biên tập viên chuyên nghiệp. Nhiệm vụ tối thượng: Viết bản tóm tắt có độ dài CHÍNH XÁC khoảng {target_words} từ. Nếu thông tin đầu vào ngắn, hãy sử dụng kỹ năng diễn giải chi tiết, phân tích sâu các mệnh đề, nối câu mạch lạc để kéo dài văn bản, nhưng TUYỆT ĐỐI KHÔNG bịa đặt thêm dữ liệu, tên người hay sự kiện ngoài văn bản."
        },
        {
            "role": "user", 
            "content": f"Yêu cầu: Viết bản tóm tắt dài khoảng {target_words} từ dựa trên dữ liệu sau:\n\n{combined_text}"
        }
    ]
    
    text_input = qwen_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = qwen_tokenizer([text_input], return_tensors="pt").to(device)
    
    # 2. Tính toán biên độ Token (1 từ Tiếng Việt ~ 1.2 -> 1.5 token)
    # Ép buộc phần cứng: Model không được phép dừng lại nếu chưa viết đủ min_out_tokens
    min_out_tokens = int(target_words * 1.1) 
    max_out_tokens = int(target_words * 1.8)
    
    print(f"[-] Qwen đang bị ép viết tối thiểu {min_out_tokens} tokens để đạt mốc ~{target_words} từ...")
    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **model_inputs,
            min_new_tokens=min_out_tokens, # Vũ khí ép buộc sinh thêm chữ
            max_new_tokens=max_out_tokens, 
            temperature=0.4,               # Tăng nhẹ lên 0.4 để văn phong diễn giải phong phú hơn
            top_p=0.9,
            repetition_penalty=1.15        # Tăng nhẹ phạt lặp từ vì văn bản dài dễ bị lặp
        )
        
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    
    return qwen_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
# ==========================================
# THỰC THI PIPELINE VỚI CẤU HÌNH TUỲ CHỈNH
# ==========================================
print("="*60)
print(" HỆ THỐNG TÓM TẮT THÔNG MINH (DYNAMIC CHUNKS) ")
print("="*60)

# ----------------------------------------------------
# ⚙️ NGƯỜI DÙNG NHẬP THÔNG SỐ TẠI ĐÂY
# ----------------------------------------------------
INPUT_JSON_FILE = '/kaggle/input/datasets/vietnguyencong123/data-test/test.json'
TARGET_SUMMARY_WORDS = int(input("Nhập số lượng từ bạn muốn tóm tắt: "))   # <-- Bạn có thể thay đổi số lượng từ muốn tóm tắt ở đây
# ----------------------------------------------------

try:
    # 1. Đọc các đoạn chunk có sẵn từ file json
    with open(INPUT_JSON_FILE, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
        
    # 2. Gom lại và chuyển về text bình thường (Xóa Markdown)
    print("[-] Đang nối và làm sạch dữ liệu đầu vào...")
    raw_text = " ".join([item.get('text', '') for item in raw_data])
    clean_text_data = clean_markdown(raw_text)
    
    # 3. Chia lại chunk cho đều (Re-chunk)
    print("[-] Đang chia lại kích thước chunk cho đồng đều (150 từ/chunk)...")
    uniform_chunks = create_uniform_chunks(clean_text_data, max_tokens=150)
    print(f"-> Đã tạo ra {len(uniform_chunks)} uniform chunks.")
    
    if len(uniform_chunks) > 0:
        # 4. Tính k động dựa trên số từ muốn tóm tắt
        dynamic_k = calculate_dynamic_k(target_words=TARGET_SUMMARY_WORDS, chunk_size=150)
        
        # Ngăn lỗi k lớn hơn tổng số chunk thực tế
        dynamic_k = min(dynamic_k, len(uniform_chunks))
        
        # Nếu bài quá ngắn, cảnh báo nhẹ cho người dùng
        if dynamic_k == len(uniform_chunks):
             print(f"[!] Chú ý: Bài báo gốc khá ngắn, hệ thống sẽ sử dụng toàn bộ bài để viết lại.")
        
        # 5. TextRank lấy top k
        top_chunks = get_top_k_chunks(uniform_chunks, k=dynamic_k)
        
        # 6. Đưa cho Qwen
        final_result = rewrite_with_qwen(top_chunks, target_words=TARGET_SUMMARY_WORDS)
        
        print("\n" + "="*60)
        print(f"🏆 BẢN TÓM TẮT HOÀN CHỈNH (~{TARGET_SUMMARY_WORDS} TỪ):")
        print("="*60)
        print(textwrap.fill(final_result, width=100))
        
except FileNotFoundError:
    print(f"[!] LỖI: Không tìm thấy file '{INPUT_JSON_FILE}'.")
except NameError as e:
    print(f"[!] Lỗi chưa tải Model: {e}. Vui lòng đảm bảo bạn đã chạy Cell 1 trước đó.")
except Exception as e:
    print(f"[!] CÓ LỖI XẢY RA: {e}")

 HỆ THỐNG TÓM TẮT THÔNG MINH (DYNAMIC CHUNKS) 


Nhập số lượng từ bạn muốn tóm tắt:  500


[-] Đang nối và làm sạch dữ liệu đầu vào...
[-] Đang chia lại kích thước chunk cho đồng đều (150 từ/chunk)...
-> Đã tạo ra 95 uniform chunks.
[-] Đang chạy TextRank để nhặt ra Top 5 chunks quan trọng nhất...
[-] Qwen đang bị ép viết tối thiểu 550 tokens để đạt mốc ~500 từ...

🏆 BẢN TÓM TẮT HOÀN CHỈNH (~500 TỪ):
Để tạo nên bản tóm tắt đầy đủ về nội dung được đưa ra, chúng ta cần chú ý đến các yếu tố chính liên
quan đến quá trình xây dựng và phát triển đất nước dưới thời nhà Mạc, cùng với vai trò của nhà Mạc
trong việc kiểm soát lãnh thổ và nhân sự.  Nhà Mạc bắt đầu xây dựng đất nước bằng việc tiếp nhận một
số quy tắc pháp chế từ nhà Lê sơ, đồng thời chỉnh sửa một chút để thích nghi với hoàn cảnh thực tế.
Họ nắm giữ một phạm vi lãnh thổ rất rộng lớn, trải dài từ Bắc xuống Nam, bao gồm 13 đạo Thừa tuyên
như Lạng Sơn, An Bang, Ninh Sóc, Hưng Hóa, Tuyên Quang, Kinh Bắc, Hải Dương, Sơn Tây, Sơn Nam, Thanh
Hóa, Nghệ An, Thuận Hóa, Quảng Nam và 1 phủ Phụng Thiên.   Sau khi nhà Mạc nắm giữ lãnh

In [ ]:
import gradio as gr
import json
import re
import torch
import networkx as nx
import math
import textwrap
from sentence_transformers import util

# ==========================================
# 1. WRAP CÁC HÀM XỬ LÝ VÀO MỘT PIPELINE
# ==========================================

def summarize_ui_pipeline(json_file, target_words):
    try:
        # Đọc file JSON từ người dùng upload
        with open(json_file.name, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
            
        # [Bước 1 & 2] Nối và làm sạch dữ liệu
        raw_text = " ".join([item.get('text', '') for item in raw_data])
        clean_text_data = clean_markdown(raw_text)
        
        # [Bước 3] Chia lại chunk
        uniform_chunks = create_uniform_chunks(clean_text_data, max_tokens=150)
        
        if not uniform_chunks:
            return "❌ Lỗi: File JSON không chứa dữ liệu văn bản hợp lệ."
            
        # [Bước 4] Tính k động
        dynamic_k = calculate_dynamic_k(target_words=target_words, chunk_size=150)
        dynamic_k = min(dynamic_k, len(uniform_chunks))
        
        # [Bước 5] TextRank
        top_chunks = get_top_k_chunks(uniform_chunks, k=dynamic_k)
        
        # [Bước 6] Qwen viết lại
        final_result = rewrite_with_qwen(top_chunks, target_words=target_words)
        
        # Format lại văn bản in ra cho đẹp
        formatted_result = textwrap.fill(final_result, width=110)
        
        return formatted_result

    except Exception as e:
        return f"❌ Có lỗi xảy ra trong quá trình xử lý: {str(e)}"

# ==========================================
# 2. XÂY DỰNG GIAO DIỆN (UI)
# ==========================================

# Thiết lập theme và layout
with gr.Blocks(theme=gr.themes.Soft(), title="AI Summarizer Pro") as demo:
    gr.Markdown("""
    # 🤖 Hệ Thống Tóm Tắt Thông Minh
    Kéo thả file **JSON** chứa các đoạn chunk và chọn số từ bạn muốn. Hệ thống sẽ tự động lọc ý chính và biên tập lại bằng **Qwen 3B**.
    """)
    
    with gr.Row():
        # Cột trái: Nhập liệu
        with gr.Column(scale=1):
            file_input = gr.File(label="Tải lên file JSON", file_types=[".json"])
            target_slider = gr.Slider(
                minimum=100, 
                maximum=1500, 
                value=500, 
                step=50, 
                label="Số lượng từ mong muốn"
            )
            submit_btn = gr.Button("🚀 Bắt đầu Tóm tắt", variant="primary")
            
        # Cột phải: Kết quả
        with gr.Column(scale=2):
            output_display = gr.Textbox(
                label="🏆 Bản tóm tắt hoàn chỉnh", 
                lines=25, 
                placeholder="Kết quả sẽ hiển thị ở đây..."
            )
            
    # Hiệu ứng Loading (AI đang suy nghĩ) được tự động kích hoạt bởi Gradio
    submit_btn.click(
        fn=summarize_ui_pipeline,
        inputs=[file_input, target_slider],
        outputs=output_display
    )

# ==========================================
# 3. KÍCH HOẠT (LAUNCH)
# ==========================================
# share=True sẽ tạo đường link công khai giúp bạn vào thẳng giao diện
demo.launch(share=True, debug=True)

/tmp/ipykernel_55/66086425.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="AI Summarizer Pro") as demo:


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://5ef97f558267068df8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[-] Đang chạy TextRank để nhặt ra Top 7 chunks quan trọng nhất...
[-] Qwen đang bị ép viết tối thiểu 770 tokens để đạt mốc ~700 từ...
[-] Đang chạy TextRank để nhặt ra Top 7 chunks quan trọng nhất...
[-] Qwen đang bị ép viết tối thiểu 770 tokens để đạt mốc ~700 từ...
